<a href="https://colab.research.google.com/github/prishaarorain-collab/Agentic-AI-in-Customer-Service-and-Sales/blob/main/Agentic_AI_Ad_Optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ad Optimization Agent

### Architecture
```
load_data → compute_metrics → llm_decide → apply_guardrails → log_decision → [loop back]
```
Each day, Claude reads the channel performance metrics and reasons about how to shift the budget. Guardrails are enforced as a separate node after Claude's decision.


## 1 · Install dependencies

In [ ]:
%pip install -q langgraph langchain-anthropic langchain-core
print("Dependencies installed")


Dependencies installed


## 2 · Configuration & System Prompt

In [ ]:
import os
import getpass
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("API key loaded from Colab Secrets")
except Exception:
    key = getpass.getpass("Enter your Anthropic API key: ")
    os.environ["ANTHROPIC_API_KEY"] = key
    print("API key set")

# Budget guardrails
TOTAL_DAILY_BUDGET = 1000.00
CHANNELS           = ["Search", "Social", "Display"]
MIN_ALLOCATION     = 0.10
MAX_DAILY_SHIFT    = 0.20
MAX_FLOOR_DAYS     = 2

# System prompt -- defines the agent role
SYSTEM_PROMPT = 'You are an expert digital advertising budget optimization agent.\n\nYour job is to allocate a fixed daily budget of $1,000 across three ad channels\n(Search, Social, Display) to maximize total conversions.\n\nEach day you will receive:\n- Each channel\'s conversion rate (CVR = conversions / clicks)\n- Each channel\'s click-through rate (CTR = clicks / impressions)\n- Each channel\'s cost per acquisition (CPA = spend / conversions)\n- The current budget allocation (% per channel)\n\nYou must respond with ONLY a JSON object in this exact format:\n{\n  "Search": <number between 0 and 1>,\n  "Social": <number between 0 and 1>,\n  "Display": <number between 0 and 1>,\n  "rationale": "<one sentence explaining your decision>"\n}\n\nRules you must follow:\n- The three numbers must sum to exactly 1.0\n- No channel should receive less than 0.10 (10%) -- every channel must stay active\n- Shift budget toward channels with higher CVR\n- Do not make sudden extreme changes -- prefer gradual shifts\n- Keep your rationale concise and data-driven (mention which metric drove the decision)'

print(f"   Budget: ${TOTAL_DAILY_BUDGET:,.0f}/day across {CHANNELS}")
print(f"   Min allocation: {MIN_ALLOCATION*100:.0f}% | Max shift: +/-{MAX_DAILY_SHIFT*100:.0f}%")


API key loaded from Colab Secrets
   Budget: $1,000/day across ['Search', 'Social', 'Display']
   Min allocation: 10% | Max shift: +/-20%


## 3 · Generate Mock Data (21 days × 3 channels)

In [ ]:
import csv, json, os
from collections import defaultdict

# ── Upload ad_data.csv when prompted ──────────────────────────────────────────
try:
    from google.colab import files
    print("📂 Please upload your ad_data.csv file...")
    uploaded = files.upload()
    CSV_FILE = list(uploaded.keys())[0]
    print(f"✅ Uploaded: {CSV_FILE}")
except ImportError:
    # Running locally — set the path to your CSV here
    CSV_FILE = "ad_data.csv"
    print(f"📂 Running locally — using: {CSV_FILE}")

# ── Load into dict keyed by date ───────────────────────────────────────────────
def load_data(path):
    data = defaultdict(list)
    with open(path, newline="") as f:
        for row in csv.DictReader(f):
            row["spend"]       = float(row["spend"])
            row["impressions"] = int(row["impressions"])
            row["clicks"]      = int(row["clicks"])
            row["conversions"] = int(row["conversions"])
            data[row["date"]].append(row)
    return dict(sorted(data.items()))

data_by_date = load_data(CSV_FILE)

import pandas as pd
df = pd.read_csv(CSV_FILE)
print(f"✅ Loaded {len(df)} rows — {df['date'].nunique()} days, {df['channel'].nunique()} channels")
df.head(9)

📂 Please upload your ad_data.csv file...


Saving ad_data.csv to ad_data (1).csv
✅ Uploaded: ad_data (1).csv
✅ Loaded 63 rows — 21 days, 3 channels


,date,channel,spend,impressions,clicks,conversions
0,2024-01-01,Search,333.33,12000,480,24
1,2024-01-01,Social,333.33,25000,375,11
2,2024-01-01,Display,333.33,40000,200,6
3,2024-01-02,Search,333.33,11800,472,23
4,2024-01-02,Social,333.33,24500,368,10
5,2024-01-02,Display,333.33,41000,205,5
6,2024-01-03,Search,333.33,12200,488,25
7,2024-01-03,Social,333.33,25500,383,12
8,2024-01-03,Display,333.33,39500,198,6


## 4 · Build the LangGraph Agent

### Graph structure
```
compute_metrics → llm_decide → apply_guardrails → log_decision
       ↑___________________________________|  (loops per day)
```

Each node does one job:
- **`compute_metrics`** — reads today's CSV rows, calculates CVR/CTR/CPA
- **`llm_decide`** — sends metrics to Claude, gets back a JSON allocation + rationale
- **`apply_guardrails`** — clamps Claude's output to ±20% shift, 10% floor
- **`log_decision`** — records the final allocation and moves to the next day


In [ ]:
import json, re
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import SystemMessage, HumanMessage

# ── Agent state — passed between nodes ────────────────────────────────────────
class AgentState(TypedDict):
    dates: list           # all dates in order
    day_index: int        # which day we're on
    current_alloc: dict   # current fractional allocation {channel: float}
    floor_counts: dict    # consecutive days each channel has been at floor
    metrics: dict         # today's computed metrics
    raw_llm_alloc: dict   # what Claude proposed (before guardrails)
    final_alloc: dict     # after guardrails applied
    rationale: str        # Claude's explanation
    decisions: list       # accumulated log of all decisions

# ── LLM client ────────────────────────────────────────────────────────────────
llm = ChatAnthropic(model="claude-sonnet-4-5", max_tokens=512, temperature=0)

# ─────────────────────────────────────────────────────────────────────────────
# NODE 1: compute_metrics
# Reads today's data rows and calculates CVR, CTR, CPA per channel
# ─────────────────────────────────────────────────────────────────────────────
def compute_metrics(state: AgentState) -> AgentState:
    date = state["dates"][state["day_index"]]
    rows = data_by_date[date]
    metrics = {}
    for row in rows:
        ch = row["channel"]
        cvr = row["conversions"] / row["clicks"]      if row["clicks"] > 0      else 0.0
        ctr = row["clicks"]      / row["impressions"] if row["impressions"] > 0 else 0.0
        cpa = row["spend"]       / row["conversions"] if row["conversions"] > 0 else 999.0
        metrics[ch] = {
            "cvr": round(cvr, 4),
            "ctr": round(ctr, 4),
            "cpa": round(cpa, 2),
            "conversions": row["conversions"],
            "clicks":      row["clicks"],
            "spend":       row["spend"],
        }
    return {**state, "metrics": metrics}

# ─────────────────────────────────────────────────────────────────────────────
# NODE 2: llm_decide
# Sends metrics + current allocation to Claude, gets back JSON allocation
# ─────────────────────────────────────────────────────────────────────────────
def llm_decide(state: AgentState) -> AgentState:
    date    = state["dates"][state["day_index"]]
    metrics = state["metrics"]
    alloc   = state["current_alloc"]

    # Build the human message with today's performance data
    metrics_text = "\n".join(
        f"  {ch}: CVR={m['cvr']:.2%}  CTR={m['ctr']:.2%}  CPA=${m['cpa']:.2f}  conversions={m['conversions']}"
        for ch, m in metrics.items()
    )
    alloc_text = "  " + ", ".join(f"{ch}: {v*100:.1f}%" for ch, v in alloc.items())

    user_msg = f"""Date: {date}

Today's channel performance:
{metrics_text}

Current budget allocation:
{alloc_text}

Based on this data, propose tomorrow's budget allocation to maximize conversions.
Respond ONLY with valid JSON — no markdown, no explanation outside the JSON."""

    # Day 1 — no prior data, just ask for equal split
    if state["day_index"] == 0:
        result = {"Search": 0.333, "Social": 0.333, "Display": 0.334,
                  "rationale": "Day 1: starting with equal split — no prior performance data available."}
    else:
        response = llm.invoke([SystemMessage(content=SYSTEM_PROMPT),
                               HumanMessage(content=user_msg)])
        text = response.content.strip()
        # Strip markdown fences if Claude wraps in ```json
        text = re.sub(r"^```(?:json)?\n?", "", text)
        text = re.sub(r"\n?```$", "", text)
        try:
            result = json.loads(text)
        except json.JSONDecodeError as e:
            print(f"   ⚠️ JSON parse error: {e}")
            print(f"   Raw response: {repr(text[:300])}")
            # Fallback: keep current allocation unchanged
            result = {**alloc, "rationale": f"Parse error — keeping current allocation."}

    return {**state, "raw_llm_alloc": result, "rationale": result.get("rationale", "")}

# ─────────────────────────────────────────────────────────────────────────────
# NODE 3: apply_guardrails
# Clamps Claude's proposed allocation to ±20% shift and 10% floor
# ─────────────────────────────────────────────────────────────────────────────
def apply_guardrails(state: AgentState) -> AgentState:
    proposed = state["raw_llm_alloc"]
    current  = state["current_alloc"]
    floors   = state["floor_counts"]
    clamped  = {}

    for ch in CHANNELS:
        raw    = proposed.get(ch, 1/3)
        prev   = current.get(ch, 1/3)
        # Cap the change at MAX_DAILY_SHIFT
        delta  = max(-MAX_DAILY_SHIFT, min(MAX_DAILY_SHIFT, raw - prev))
        clamped[ch] = prev + delta

    # Enforce minimum floor
    for ch in CHANNELS:
        clamped[ch] = max(clamped[ch], MIN_ALLOCATION)

    # Normalize so allocations sum to 1.0
    total = sum(clamped.values())
    clamped = {ch: round(v / total, 4) for ch, v in clamped.items()}

    # Update floor counters (how many consecutive days at the minimum)
    new_floors = {}
    for ch in CHANNELS:
        if clamped[ch] <= MIN_ALLOCATION + 0.005:
            new_floors[ch] = floors.get(ch, 0) + 1
        else:
            new_floors[ch] = 0

    return {**state, "final_alloc": clamped, "floor_counts": new_floors}

# ─────────────────────────────────────────────────────────────────────────────
# NODE 4: log_decision
# Records the final decision and advances to the next day
# ─────────────────────────────────────────────────────────────────────────────
def log_decision(state: AgentState) -> AgentState:
    date    = state["dates"][state["day_index"]]
    alloc   = state["final_alloc"]
    metrics = state["metrics"]

    spend   = {ch: round(alloc[ch] * TOTAL_DAILY_BUDGET, 2) for ch in CHANNELS}

    decision = {
        "date":       date,
        "allocation": alloc,
        "spend_$":    spend,
        "metrics":    metrics,
        "rationale":  state["rationale"],
    }

    # Print daily summary
    alloc_str = " | ".join(f"{ch} ${spend[ch]:.0f} ({alloc[ch]*100:.0f}%)" for ch in CHANNELS)
    print(f"\n📅 {date}")
    print(f"   {alloc_str}")
    print(f"   💬 {state['rationale']}")

    return {
        **state,
        "current_alloc": alloc,
        "decisions": state["decisions"] + [decision],
        "day_index": state["day_index"] + 1,
    }

# ─────────────────────────────────────────────────────────────────────────────
# ROUTING — keep looping until all days are processed
# ─────────────────────────────────────────────────────────────────────────────
def should_continue(state: AgentState) -> str:
    if state["day_index"] < len(state["dates"]):
        return "continue"
    return "done"

# ─────────────────────────────────────────────────────────────────────────────
# BUILD THE GRAPH
# ─────────────────────────────────────────────────────────────────────────────
builder = StateGraph(AgentState)
builder.add_node("compute_metrics",  compute_metrics)
builder.add_node("llm_decide",       llm_decide)
builder.add_node("apply_guardrails", apply_guardrails)
builder.add_node("log_decision",     log_decision)

builder.set_entry_point("compute_metrics")
builder.add_edge("compute_metrics",  "llm_decide")
builder.add_edge("llm_decide",       "apply_guardrails")
builder.add_edge("apply_guardrails", "log_decision")
builder.add_conditional_edges("log_decision", should_continue,
                               {"continue": "compute_metrics", "done": END})

graph = builder.compile()
print("✅ LangGraph agent compiled")
print("   Nodes: compute_metrics → llm_decide → apply_guardrails → log_decision → [loop]")


✅ LangGraph agent compiled
   Nodes: compute_metrics → llm_decide → apply_guardrails → log_decision → [loop]


## 5 · Run the Agent

In [ ]:
dates = sorted(data_by_date.keys())

initial_state = AgentState(
    dates        = dates,
    day_index    = 0,
    current_alloc= {ch: 1/3 for ch in CHANNELS},
    floor_counts = {ch: 0   for ch in CHANNELS},
    metrics      = {},
    raw_llm_alloc= {},
    final_alloc  = {},
    rationale    = "",
    decisions    = [],
)

print("=" * 65)
print("AD OPTIMIZATION AGENT — RUNNING")
print("=" * 65)

final_state = graph.invoke(initial_state)
decisions   = final_state["decisions"]

# Save decision log
with open("agent_decisions.jsonl", "w") as f:
    for d in decisions:
        f.write(json.dumps(d) + "\n")

print(f"\n{'='*65}")
print(f"✅ Done — {len(decisions)} days processed")

AD OPTIMIZATION AGENT — RUNNING

📅 2024-01-01
   Search $333 (33%) | Social $333 (33%) | Display $334 (33%)
   💬 Day 1: starting with equal split — no prior performance data available.

📅 2024-01-02
   Search $500 (50%) | Social $300 (30%) | Display $200 (20%)
   💬 Search has the highest CVR at 4.87% and lowest CPA at $14.49, so increasing its allocation while maintaining minimum thresholds for other channels.

📅 2024-01-03
   Search $550 (55%) | Social $300 (30%) | Display $150 (15%)
   💬 Search has the highest CVR at 5.12% and lowest CPA at $13.33, so increasing its allocation while reducing Display's underperforming 3.03% CVR.

📅 2024-01-04
   Search $600 (60%) | Social $270 (27%) | Display $130 (13%)
   💬 Search has the highest CVR at 5.19% and lowest CPA at $13.58, so increasing its allocation while maintaining minimum thresholds on other channels.

📅 2024-01-05
   Search $650 (65%) | Social $250 (25%) | Display $100 (10%)
   💬 Search has the highest CVR at 5.30% and lowest CPA at

## 6 · Evaluation - Agent vs Equal-Split Baseline

Projects conversions for both strategies using the same channel CVRs but different budget splits.


In [ ]:
import pandas as pd

rows = []
for d in decisions:
    metrics = d["metrics"]
    agent_conv   = 0
    baseline_conv = 0
    for ch, m in metrics.items():
        clicks_per_dollar = m["clicks"] / m["spend"] if m["spend"] > 0 else 0
        # Agent projected conversions
        agent_spend  = d["spend_$"].get(ch, 0)
        agent_conv  += agent_spend * clicks_per_dollar * m["cvr"]
        # Baseline: equal split
        base_spend   = TOTAL_DAILY_BUDGET / len(CHANNELS)
        baseline_conv += base_spend * clicks_per_dollar * m["cvr"]

    rows.append({
        "date":          d["date"],
        "agent_conv":    round(agent_conv, 1),
        "baseline_conv": round(baseline_conv, 1),
        "lift_pct":      round((agent_conv - baseline_conv) / max(baseline_conv, 1) * 100, 1),
    })

df_eval = pd.DataFrame(rows)
total_agent    = df_eval["agent_conv"].sum()
total_baseline = df_eval["baseline_conv"].sum()
total_spend    = len(decisions) * TOTAL_DAILY_BUDGET
overall_lift   = (total_agent - total_baseline) / total_baseline * 100
agent_cpa      = total_spend / total_agent
baseline_cpa   = total_spend / total_baseline

print("=" * 55)
print("EVALUATION: AGENT vs EQUAL-SPLIT BASELINE")
print("=" * 55)
print(f"{'Date':<13} {'Agent':>8} {'Baseline':>10} {'Lift':>8}")
print("-" * 42)
for _, r in df_eval.iterrows():
    print(f"{r['date']:<13} {r['agent_conv']:>8.1f} {r['baseline_conv']:>10.1f} {r['lift_pct']:>7.1f}%")
print("-" * 42)
print(f"{'TOTAL':<13} {total_agent:>8.1f} {total_baseline:>10.1f} {overall_lift:>7.1f}%")
print()
print(f"  Agent avg CPA    : ${agent_cpa:.2f}")
print(f"  Baseline avg CPA : ${baseline_cpa:.2f}")
print(f"  CPA improvement  : ${baseline_cpa - agent_cpa:.2f} saved per conversion")

best_channel = max(decisions[-1]["allocation"], key=decisions[-1]["allocation"].get)
print(f"Agent projection: {total_agent:.0f} conversions vs baseline {total_baseline:.0f} — a {overall_lift:.1f}% lift.")
print(f"Best channel: {best_channel}. Agent avg CPA ${agent_cpa:.2f} vs baseline avg CPA ${baseline_cpa:.2f}.")
df_eval

EVALUATION: AGENT vs EQUAL-SPLIT BASELINE
Date             Agent   Baseline     Lift
------------------------------------------
2024-01-01        41.0       41.0    -0.1%
2024-01-02        46.5       38.0    22.3%
2024-01-03        54.7       43.0    27.3%
2024-01-04        55.5       41.7    33.2%
2024-01-05        59.4       42.5    39.9%
2024-01-06        62.2       42.1    47.8%
2024-01-07        65.9       42.8    54.0%
2024-01-08        69.3       43.6    58.8%
2024-01-09        70.2       43.1    62.8%
2024-01-10        69.8       43.0    62.2%
2024-01-11        71.4       43.8    63.1%
2024-01-12        70.8       45.2    56.8%
2024-01-13        73.1       48.0    52.4%
2024-01-14        75.3       49.9    50.8%
2024-01-15        77.1       51.5    49.9%
2024-01-16        79.1       53.1    48.9%
2024-01-17        78.8       53.8    46.5%
2024-01-18        80.6       55.4    45.6%
2024-01-19        80.3       56.0    43.4%
2024-01-20        82.2       57.6    42.6%
2024-01-21  

,date,agent_conv,baseline_conv,lift_pct
0,2024-01-01,41.0,41.0,-0.1
1,2024-01-02,46.5,38.0,22.3
2,2024-01-03,54.7,43.0,27.3
3,2024-01-04,55.5,41.7,33.2
4,2024-01-05,59.4,42.5,39.9
5,2024-01-06,62.2,42.1,47.8
6,2024-01-07,65.9,42.8,54.0
7,2024-01-08,69.3,43.6,58.8
8,2024-01-09,70.2,43.1,62.8
9,2024-01-10,69.8,43.0,62.2
